# Where this all fits

> A working model in eight lines of code, the handful of words you will hear over and over, and why almost none of this worked until recently.

Read this chapter at `/learn/01-where-this-fits/`. Exported from `src/content/chapters/01-where-this-fits.mdx` — edit there, not here.


Before we explain anything at all, here is a machine learning model. You do
not need to understand a word of it yet. Press **Run** and see what happens.

In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

digits = load_digits()                       # 1797 hand-written digits, 8x8 pixels
X_train, X_valid, y_train, y_valid = train_test_split(
    digits.data, digits.target, test_size=0.3, random_state=0)

model = LogisticRegression(max_iter=5000).fit(X_train, y_train)
print(f"accuracy on digits it has never seen: {model.score(X_valid, y_valid):.1%}")

The first time takes a few seconds, because your browser is quietly downloading
a whole copy of Python. After that it's instant.

<Says who="juno">
If you skipped the button: go back and press it. This site is built around
running things, and reading about a result is not the same as watching one
appear. Nothing installs, nothing breaks, and you can press it as many times as
you like.
</Says>

What just happened: the computer read handwriting. Somebody scanned in 1797
handwritten digits; the program was shown about seventy per cent of them along
with the right answers, and then tested on the rest — digits it had never seen.
It got roughly nineteen out of twenty right.

The part worth sitting with is this one.

**Nobody told it what a 7 looks like.**

Nowhere in those eight lines does anyone describe a pen stroke, or deal with the
7s that have a crossbar and the 7s that don't, or mention the little hook some
people put on a 9. There is no instruction about digits anywhere in there. There
isn't a single *fact* about digits anywhere in there. And it reads handwriting
anyway.

Nobody wrote the rules. Somebody handed over examples, and a program went and
found the rules. That swap is what the whole subject is about. Everything in the
other fifteen chapters is the machinery for doing it.

## The inversion

Here is a shape worth having in front of you:

<div class="table-scroll">

| | You provide | The computer gives back |
|---|---|---|
| **Ordinary program** | rules + input | output |
| **Machine learning** | input + output | **rules** |

</div>

Read the second row again. You hand over the questions *and* the answers, and
what comes back is the method.

That is an odd sort of trade, and it is only worth making for a particular kind
of problem: the kind where you cannot say what the rule is.

Here is the difference, and you can test it on yourself right now. Think of how
you sort your post into "keep" and "bin". You could probably write that down —
these envelopes, from these senders, in these months. It would be tedious, but
you could do it.

Now think about how you know a photograph has a cat in it. Not *whether* it does
— you can tell in about a twentieth of a second without trying. Try to write down
the steps. What exactly do you look for first? How do you handle a cat that's
mostly behind a chair? A black cat in a dark room? A cat from behind?

You can't write it down. Not because you don't know a cat when you see one, but
because the knowing isn't in a form you can spell out. It's in there and it won't
come out in words.

Machine learning is, mostly, a set of tools for getting knowledge out of that
locked room by showing examples through the window.

The closest thing you already write is the gap between a function and a
*constraint*. You know the signature you want:

`fn is_seven(pixels: &[u8]) -> bool`

You have no idea what goes in the body, and no amount of staring will produce
it. What you *can* produce is a pile of `(input, expected)` pairs and a way to
score how badly a candidate body is doing. Machine learning fills in the body by
searching, guided by that score.

## The four words, once, and then we'll stop

Four words get thrown around in public as if they mean the same thing. They
don't. Five minutes here saves you a lot of confusion later, and then we mostly
won't need them again.

**Artificial intelligence** is the whole field, and it's old — the name was
coined at a workshop in 1956. It covers chess engines built entirely from
hand-written search, expert systems built from hand-written rules, and everything
below. It's an *aspiration*, not a technique. Almost nobody describes their own
work as "AI" when talking to another practitioner.

**Machine learning** is the part of AI where the rules get fitted to data
instead of written by hand. Logistic regression is machine learning. So is a
decision tree. So was the spam filter that shipped in 2002. Most of it involves
no neural networks at all.

**Deep learning** is the part of machine learning that uses neural networks with
many layers. "Deep" literally means "has a lot of layers stacked up." That's the
whole etymology. It's the part that got startlingly good after about 2012, and
it's where nearly all the recent noise lives.

**A model** is the fitted thing itself — the function with its numbers filled in.
`model` in the code above is a model. A file of weights is a model. When someone
says "we deployed the model," they shipped a function.

An awkward consequence of that nesting: a straight-line fit from 1805 is,
technically and correctly, artificial intelligence. Which means "AI-powered" is a
claim entirely compatible with a spreadsheet formula.

Being able to ask *which layer of that nest do you mean?* — politely — turns out
to be quite useful.

<Says who="basil">
Worth knowing while you read anything written for the public: the four words
above get used interchangeably by people who should know better, and sometimes by
people who benefit from the confusion. When a product says it uses AI, it may
mean a language model, and it may mean an average.
</Says>

Three more words and we can get back to the interesting part:

- **Training** — finding the numbers. Expensive, done once, offline.
- **Inference** — running the finished model on something new. Cheap, done constantly.
- **Parameters** (or *weights*) — the numbers that got found. The model above found
  650 of them. The big language models have hundreds of billions.

## Let's take it apart

I don't want you to take my word for any of this, so let's open the model up. It
will not take long, because there is almost nothing inside.

In [ ]:
w = model.coef_          # the learned weights
b = model.intercept_     # the learned offsets
print("weights:", w.shape, " offsets:", b.shape)
print("total learned numbers:", w.size + b.size)

Ten rows of 64 weights, plus ten offsets. One row per digit, one weight per
pixel. That's it. That is the entire model — 650 floating-point numbers.
Everything it learned from 1257 examples of human handwriting is sitting in
those 650 numbers and nowhere else.

To decide what an image is, it gives each of the ten digits a score — multiply
every pixel by that digit's weight, add them all up — and picks whichever scores
highest. We can do that ourselves, without the library, so you can see there is
nothing hidden.

In [ ]:
import numpy as np

image = X_valid[0]                    # one 8x8 digit, flattened to 64 numbers
scores = image @ w.T + b              # ten scores, one per digit
print("scores :", np.round(scores, 1))
print("predicted:", scores.argmax(), " actual:", y_valid[0])

One matrix multiply, one addition, one
argmax. That is *inference* — the whole of it.

Which means the expensive part was never running the model. Running it costs a
handful of multiplications. The expensive part was finding those 650 numbers, and
that asymmetry — costly once, cheap forever after — is why any of this is
economically interesting at all.

There is a nice way to see it.

Each row of `w` is a **template** — a 64-number picture of what that digit tends
to look like. The  between an image and a template comes
out large when the bright pixels of the image line up with the large weights of
the template.

So "which digit is this?" quietly becomes "which template does this image agree
with most?", and agreement is measured by a dot product. That's all a linear
classifier ever is: a shelf of templates and a way to measure agreement.

You don't have to take that on trust, because the templates are pictures and we
can look at them:

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(2, 5, figsize=(7, 3))
for digit, ax in enumerate(axes.flat):
    ax.imshow(w[digit].reshape(8, 8), cmap="RdBu_r")
    ax.set_title(str(digit), fontsize=9); ax.axis("off")
plt.tight_layout()

Red means "a bright pixel here argues *for* this digit." Blue means it argues
against.

Look at the 0: it's a ring, and the middle is blue. The model worked out that a
zero is a shape with a *hole*, and that ink in the middle is evidence against.
Look at the 1: a vertical stripe. Nobody wrote either of those down. They fell
out of 1257 examples and some arithmetic.

And here — right here — is the limitation, visible in the same picture. One
template per class cannot say "a 7 with a crossbar *or* a 7 without one." It has
to average them, and the average of two different-looking 7s is a blurry 7 that
matches neither very well. You can see it in the plot if you look: the harder
digits are muddier.

Fixing exactly that is what layers are for, and that's
[Chapter 8](/learn/08-neural-networks/). Keep this picture in mind until then.

**"I ran it and got a slightly different number."** Good — that means you're
paying attention. Small differences come from library versions and the browser's
maths. Anything in the 94–96% range is the same result. If you get something
wildly different, something is wrong and I'd want to know.

**"What is that `@` symbol?"** It multiplies two grids of numbers together in
the particular way described just above — every row against every column. Python
has a dedicated symbol for it because this operation turns up constantly here.
Hover the underlined matrix multiply for a short
explanation, and there's a longer one in the appendix.

**"I don't see how multiplying pixels by numbers 'recognises' anything."** You're
not missing something — that reaction is correct and it's the right thing to be
suspicious of. Open the fold above and *look at the templates*. That's the
moment it usually clicks: the numbers are a picture, and the multiplication is
asking "how much does this look like that?"

**"Where did the 650 numbers come from?"** We are deliberately not answering
that yet — it's chapters 4 and 5. For today the thing to hold onto is that they
are *just numbers*. A model is a recipe someone's program found, not a mind
somebody summoned.

## So why is all this suddenly everywhere?

Machine learning has been in the news for about a decade. It is reasonable to
assume that something was invented about a decade ago. Mostly, that isn't what
happened.

**The ideas are old.**

The perceptron — a single artificial neuron, trained by exactly the method you'll
build in chapter 5 — was built as *physical hardware* in 1958. Backpropagation,
which makes deep networks trainable and which we'll write by hand in
[Chapter 9](/learn/09-backpropagation/), was popularised in 1986. Convolutional
networks were in production reading cheques for the US Postal Service in the
early 1990s. LSTMs, which handled sequences for the next two decades, are from
1997.

So "somebody finally thought of it" isn't the answer. The thinking was done
decades ago. Three other things had to turn up first.

**Data.** A model fits itself to examples, so it needs examples, and for most of
this history nobody had any. The dataset that broke the field open was ImageNet:
14 million labelled photographs, assembled between 2007 and 2009, in large part
by paying people on Mechanical Turk to label images one at a time. It took the
internet to make collection cheap and crowdsourcing to make labelling cheap.
Before that, nobody on Earth had a million labelled anything.

**Compute — and specifically the right *shape* of compute.** You saw above that
inference is a matrix multiply. Training is an enormous number of them. And it
turns out somebody had already built a machine for doing vast numbers of parallel
multiplications and additions at once, for reasons entirely to do with drawing
triangles quickly in video games. Graphics cards. When people started running
neural networks on graphics cards around 2009, the same experiment ran roughly
fifty times faster.

That number matters more than it first looks. An idea that takes six months to
test is not an idea you can tinker with. An idea that takes three days is.
Fifty times faster didn't only speed the work up; it changed which experiments
were worth trying at all.

**A pile of small, unglamorous fixes.** Better starting values for the numbers.
A simpler bend in the middle of the network, so the training signal survives many
layers ([why](/appendix/math/#relu)). Dropout, batch normalisation, Adam — all of
which you will meet later, and none of which is more than a paragraph of
arithmetic on its own. Together they moved deep networks from "should work in
principle" to "works on a Tuesday afternoon".

All three arrived at once in **September 2012**, when a network called AlexNet
won an image-recognition competition, getting 15.3% of the pictures wrong against
the runner-up's 26.2%. On a competition where everyone had been fighting over
fractions of a percent for years, a ten-point gap was enough to change what the
whole field worked on.

There's a pattern in that story, and it repeats.

Nearly every "breakthrough" here is an **old idea that finally became affordable**.
Transformers (2017) took attention, an idea from 2014, and made it cheap enough
to build big. Diffusion models are 2015 mathematics that became practical around
2021. Neural networks themselves are a 1950s idea that had to wait sixty years
for the hardware.

So when somebody says a thing in this field is impossible, it is worth asking
whether they mean impossible or merely expensive at the moment. Those two have
quite different futures.

## When the answer is not machine learning

This belongs early rather than late, because it is where the judgement lives and
almost nobody teaches it.

**When you can just write the rule.** Tax calculation is defined in legislation.
Don't learn it from examples. You'll achieve 99.4% accuracy on a problem where
100% was sitting right there for free, and you won't be able to explain the 0.6%
to anyone, least of all a regulator.

**When being wrong is unacceptable *and* unexplainable.** These models are
statistical. They are sometimes wrong and they often can't tell you why. If a
wrong answer means a wrong medication, the model is at best an input to a human
decision, and should be built and described that way.

**When you have no data.** Two hundred examples of a rare event is not a training
set, it's an anecdote. Sometimes the correct project is spending six
months building the pipeline that collects the data, and shipping the model next
year. That's a much less fun sentence to say in a planning meeting, and it is
often the right one.

**When a simple sum would do.** A surprising amount of what gets sold as "AI"
is an average, or a count, with a better name. Try the boring version first and
*measure how well it does*. That number is your baseline. If the clever model
can't beat it, you found that out in an afternoon rather than three months in.

The expensive mistake is almost never picking the wrong kind of model. It is
spending four months building one for a problem where the data could never have
answered the question in the first place.

The skill that prevents that is knowing how to frame a problem, and it's the
whole of [Chapter 3](/learn/03-the-shape-of-problems/).

## The map

Everything so far is one branch of something larger. Here is the shape of the
whole thing.

Most of it won't mean anything yet, and that's expected. It is here so that you
have somewhere to put each new idea as it arrives, rather than meeting each one
on its own with no context. Come back after every part and watch it fill in.

Three questions hang off machine learning, and running them together is the
single biggest source of early confusion:

*What feedback do you get?* (supervised, unsupervised, self-supervised,
reinforcement.) *What shape is the learned function?* (linear model, tree, neural
network.) *What makes the fitting work?* (losses, gradients, validation.)

Every project answers all three, and the answers are independent of each other.
A tree can be supervised or not; a neural network can be trained on labelled
examples or on raw data. Keeping the three questions apart makes a lot of writing
about this field much easier to follow.

Think of a decision somebody makes over and over — at your work, in a hobby, in
a household. Sorting applications, marking homework, deciding which emails
matter, spotting a bad batch on a production line. Then ask which of these three
it is:

1. A rule somebody could write down, and has.
2. A rule somebody could write down, but it's a growing tangle of exceptions and
   special cases that nobody quite trusts any more.
3. A rule nobody can write down at all, so it stays a judgement call.

The second one is the interesting category, and it is more common than people
expect. Look for anything where the instructions have a lot of "except when"
in them.

There's no right answer here — the point is the habit of asking.

Categories 2 and 3 are where machine learning earns its keep, and 2 is the one
people miss. A growing tangle of exceptions is a rule being fitted to examples
by hand, slowly, by a person. That is the same job as machine learning, done the
hard way.

Category 1 is where most wasted effort goes. If the rule can be written down and
it doesn't keep changing, write the rule.

Hold onto whichever example you picked. In chapter 3 we'll give it a shape, and
by chapter 16 you'll be able to say whether it was worth doing.

## Where you're going

By the end of the fortnight you will have built, from scratch and with nothing
clever hidden inside it: a model that fits a straight line, the search procedure
that finds its numbers, a neural network, and the method that trains one.

Then you'll build the whole lot again using a library called PyTorch, in about a
tenth of the code. Doing it the long way first is the point. When something goes
wrong later — and it will — you'll know what the library is doing on your behalf,
which is the difference between fixing a problem and guessing at it.

You'll also be able to open a paper, read the first page, and know which boxes on
that map it lives in. That skill, more than any particular architecture, is what
makes the rest of this field learnable without anybody's help.

Tomorrow: Python, at the speed of somebody who already knows how to program.